<a href="https://colab.research.google.com/github/RashidEg/-/blob/main/%D0%AD%D0%B3%D0%B0%D0%BC%D0%BE%D0%B2_%D0%90_%D0%94_%D0%9A%D0%A2%E2%84%966.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pytest pytest-asyncio aiohttp pydantic allure-pytest
!wget https://github.com/allure-framework/allure2/releases/download/2.24.1/allure-2.24.1.tgz
!tar -zxvf allure-2.24.1.tgz -C /opt/
!ln -s /opt/allure-2.24.1/bin/allure /usr/bin/allure

--2026-11-04 19:52:47--  https://github.com/allure-framework/allure2/releases/download/2.24.1/allure-2.24.1.tgz
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/59838788/11650dbc-ad10-4875-93d2-72eebecb8c48?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-04T20%3A37%3A06Z&rscd=attachment%3B+filename%3Dallure-2.24.1.tgz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-11-04T19%3A36%3A36Z&ske=2026-11-04T20%3A37%3A06Z&sks=b&skv=2018-11-09&sig=OkcxU0r%2BVgTJyjRUO2w4zfyCTThbhuuLz1h%2BfCkxadc%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MzgyMzc2NywibmJmIjoxNzkzODIxOTY3LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZH

In [ ]:
%%writefile test_petstore_async.py
import pytest
import aiohttp
import allure

BASE_URL = "https://petstore.swagger.io/v2"

PET_ID = 777888
ORDER_ID = 555
USERNAME = "test_user_6"

pet_payload = {"id": PET_ID, "name": "Balu", "status": "available"}
store_payload = {"id": ORDER_ID, "petId": PET_ID, "quantity": 1, "status": "placed", "complete": True}
user_payload = {"id": 101, "username": USERNAME, "firstName": "Ivan", "userStatus": 1}

# Общая фикстура
@pytest.fixture(scope="module")
def event_loop():
    import asyncio
    loop = asyncio.get_event_loop_policy().new_event_loop()
    yield loop
    loop.close()

@allure.feature("Pet Management")
class TestPet:
    @pytest.mark.asyncio
    @allure.story("Create a new pet")
    @allure.severity(allure.severity_level.CRITICAL)
    async def test_create_pet_positive(self):
        async with aiohttp.ClientSession() as session:
            async with session.post(f"{BASE_URL}/pet", json=pet_payload) as response:
                assert response.status == 200

    @pytest.mark.asyncio
    @allure.story("Create a new pet")
    @allure.severity(allure.severity_level.NORMAL)
    async def test_create_pet_negative(self):
        async with aiohttp.ClientSession() as session:
            # Неверный тип данных
            async with session.post(f"{BASE_URL}/pet", data="bad_data", headers={"Content-Type": "application/json"}) as response:
                assert response.status in [400, 415, 500]

    @pytest.mark.asyncio
    @allure.story("Get pet by ID")
    @allure.severity(allure.severity_level.BLOCKER)
    async def test_get_pet_positive(self):
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{BASE_URL}/pet/{PET_ID}") as response:
                assert response.status == 200

    @pytest.mark.asyncio
    @allure.story("Get pet by ID")
    @allure.severity(allure.severity_level.NORMAL)
    async def test_get_pet_negative(self):
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{BASE_URL}/pet/9999999999") as response:
                assert response.status == 404

@allure.feature("Store Management")
class TestStore:
    @pytest.mark.asyncio
    @allure.story("Place an order")
    @allure.severity(allure.severity_level.CRITICAL)
    async def test_place_order_positive(self):
        async with aiohttp.ClientSession() as session:
            async with session.post(f"{BASE_URL}/store/order", json=store_payload) as response:
                assert response.status == 200

    @pytest.mark.asyncio
    @allure.story("Place an order")
    @allure.severity(allure.severity_level.NORMAL)
    async def test_place_order_negative(self):
        bad_payload = store_payload.copy()
        bad_payload["id"] = "invalid_id"
        async with aiohttp.ClientSession() as session:
            async with session.post(f"{BASE_URL}/store/order", json=bad_payload) as response:
                assert response.status in [400, 500]

    @pytest.mark.asyncio
    @allure.story("Get order by ID")
    @allure.severity(allure.severity_level.CRITICAL)
    async def test_get_order_positive(self):
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{BASE_URL}/store/order/{ORDER_ID}") as response:
                assert response.status == 200

    @pytest.mark.asyncio
    @allure.story("Get order by ID")
    @allure.severity(allure.severity_level.NORMAL)
    async def test_get_order_negative(self):
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{BASE_URL}/store/order/0") as response:
                assert response.status == 404

@allure.feature("User Management")
class TestUser:
    @pytest.mark.asyncio
    @allure.story("Create user")
    @allure.severity(allure.severity_level.CRITICAL)
    async def test_create_user_positive(self):
        async with aiohttp.ClientSession() as session:
            async with session.post(f"{BASE_URL}/user", json=user_payload) as response:
                assert response.status == 200

    @pytest.mark.asyncio
    @allure.story("Create user")
    @allure.severity(allure.severity_level.NORMAL)
    async def test_create_user_negative(self):
        async with aiohttp.ClientSession() as session:
            async with session.post(f"{BASE_URL}/user", json={}) as response:
                assert response.status != 500

    @pytest.mark.asyncio
    @allure.story("Get user by username")
    @allure.severity(allure.severity_level.CRITICAL)
    async def test_get_user_positive(self):
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{BASE_URL}/user/{USERNAME}") as response:
                assert response.status == 200

    @pytest.mark.asyncio
    @allure.story("Get user by username")
    @allure.severity(allure.severity_level.NORMAL)
    async def test_get_user_negative(self):
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{BASE_URL}/user/invalid_username_777") as response:
                assert response.status == 404

Overwriting test_petstore_async.py


In [ ]:
!pytest test_petstore_async.py --alluredir=allure-results
!allure generate allure-results -o allure-report --clean
!zip -r allure-report.zip allure-report/

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: asyncio-1.4.1.post1, allure-pytest-2.16.2, langsmith-0.14.15, platformdirs-4.14.0, anyio-4.15.1, typeguard-4.6.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 12 items                                                             

test_petstore_async.py ............                                      [100%]

============================== 12 passed in 2.73s ==============================
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Report successfully gene